# 02 - Baseline
Un modèle en fp16 sur quelques images.

In [6]:
# Setup Colab : activer GPU (Exécution > Modifier le type d'exécution > T4)
!git clone https://github.com/Manal-o/vlm-quantization-project.git
%cd vlm-quantization-project
!pip install -q -r requirements.txt
!python data/download_flickr8k.py --n 500

Cloning into 'vlm-quantization-project'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 61 (delta 23), reused 45 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 16.58 KiB | 282.00 KiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/vlm-quantization-project/vlm-quantization-project/vlm-quantization-project
500 images enregistrées dans data/raw


In [7]:
p = 'src/models.py'
s = open(p).read()
s = s.replace(
    "from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig",
    "from transformers import AutoProcessor, BitsAndBytesConfig\n"
    "try:\n"
    "    from transformers import AutoModelForImageTextToText as AutoVLM\n"
    "except ImportError:\n"
    "    from transformers import AutoModelForVision2Seq as AutoVLM")
s = s.replace("AutoModelForVision2Seq.from_pretrained", "AutoVLM.from_pretrained")
open(p, 'w').write(s)

1980

In [8]:
import yaml, json, torch
from PIL import Image
from src.models import load, generate

cfg = yaml.safe_load(open('configs/default.yaml'))
meta = json.load(open('data/raw/metadata.json'))
proc, model = load(cfg['models'][0]['hf_id'], 'fp16')

for m in meta[:5]:
    img = Image.open(m['file']).convert('RGB')
    caption, latency = generate(proc, model, img, cfg['prompts']['caption'])
    print(f"VLM : {caption}")
    print(f"Réf : {m['references'][0]}")
    print(f"({latency:.2f} s)\n")

print(f"VRAM max : {torch.cuda.max_memory_allocated() / 1024**3:.2f} Go")

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

VLM : Two dogs are fighting in the snow.
Réf : The dogs are in the snow in front of a fence .
(1.13 s)

VLM : Man is petting dog in pool.
Réf : a brown and white dog swimming towards some in the pool
(1.02 s)

VLM : Man dancing with woman in costume in street with crowd watching.
Réf : A man and a woman in festive costumes dancing .
(1.33 s)

VLM : People sitting at a picnic table under an umbrella in front of a pub.
Réf : A couple of people sit outdoors at a table with an umbrella and talk .
(1.17 s)

VLM : A football player with the number 28 on his jersey.
Réf : A man is wearing a Sooners red football shirt and helmet .
(1.14 s)

VRAM max : 8.39 Go
